In [5]:
# compute_asteroid_params_fits_modern.py
import numpy as np
from astropy.io import fits
from astropy.table import Table
from astropy.time import Time
from astropy import units as u
from astropy.coordinates import (
    SkyCoord, EarthLocation, GCRS, CartesianRepresentation,
    solar_system_ephemeris, get_body_barycentric_posvel
)
from poliastro.bodies import Sun
from poliastro.twobody import Orbit

solar_system_ephemeris.set("builtin")

# ====== 配置 ======
INPUT_FITS = "./matched_asteroids_sbdb.fits"
OUTPUT_FITS = "asteroids_with_params.fits"

# 兴隆站 (IAU 327)
xinglong = EarthLocation(lat=40.3925*u.deg, lon=117.5766*u.deg, height=960*u.m)

# ---------- 自动解析带单位 ----------
def parse_quantity(x, default_unit=None):
    """解析 SBDB 字符串，比如 '2461000.5 d' 或者 '1.234 AU'"""
    s = str(x).strip()
    parts = s.split()
    if len(parts) == 2:
        val, unit = parts
        return float(val) * u.Unit(unit)
    else:
        return float(s) * default_unit

def parse_epoch_jd(x):
    """解析像 '2461000.5 d'、'2459000.5 jd' 这样的 epoch"""
    s = str(x).lower().replace("jd","").replace("d","")
    return Time(float(s), format="jd")

# ========== 读取 FITS ==========
tbl = Table.read(INPUT_FITS)

col_e     = "orbit_elements_e"
col_a     = "orbit_elements_a"
col_i     = "orbit_elements_i"
col_Omega = "orbit_elements_om"
col_omega = "orbit_elements_w"
col_M     = "orbit_elements_ma"
col_epoch = "orbit_epoch"

col_mjd = "MJD"
col_dateobs = "DATEOBS"

# 输出数组
r_list = []
delta_list = []
phase_list = []
ang_deg_day_list = []
ang_arcsec_hour_list = []

# ========== 主循环 ==========
for row in tbl:

    # ---- 解析轨道根数 ----
    a = parse_quantity(row[col_a], u.AU)
    e = parse_quantity(row[col_e], u.one)          # 必须是 Quantity
    inc = parse_quantity(row[col_i], u.deg)
    raan = parse_quantity(row[col_Omega], u.deg)
    argp = parse_quantity(row[col_omega], u.deg)
    M = parse_quantity(row[col_M], u.deg)

    # ---- 解析 epoch ----
    t_epoch = parse_epoch_jd(row[col_epoch])

    # ---- 观测时间 ----
    if not np.isnan(row[col_mjd]):
        t_obs = Time(row[col_mjd], format="mjd")
    else:
        t_obs = Time(row[col_dateobs])

    # ---- 从 classical elements 构造轨道 ----
    orbit_epoch = Orbit.from_classical(
        Sun, a, e, inc, raan, argp, M, epoch=t_epoch
    )

    # ---- propagate 到观测时间 ----
    orbit_obs = orbit_epoch.propagate(t_obs - t_epoch)

    # ---- 小行星在 heliocentric 坐标 ----
    r_vec = orbit_obs.r.to(u.AU).value   # ndarray shape (3,)
    r_mag = np.linalg.norm(r_vec) * u.AU

    # ---- 兴隆站在太阳系质心中的位置 ----
    earth_pos = get_body_barycentric_posvel("earth", t_obs)[0].xyz.to(u.AU).value
    obs_itrs = xinglong.get_itrs(t_obs).transform_to(GCRS(obstime=t_obs)).cartesian.xyz.to(u.AU).value
    obs_bary = earth_pos + obs_itrs

    # ---- Δ(观测者-小行星) ----
    delta_vec = r_vec - obs_bary
    delta_mag = np.linalg.norm(delta_vec) * u.AU

    # ---- 相位角 ----
    cos_alpha = np.dot(r_vec, delta_vec) / (np.linalg.norm(r_vec) * np.linalg.norm(delta_vec))
    cos_alpha = np.clip(cos_alpha, -1, 1)
    phase = np.arccos(cos_alpha) * u.rad

    # ---- 数值角速度（±1min） ----
    dt_small = 60 * u.s

    def pos_bary(t):
        orb = orbit_epoch.propagate(t - t_epoch)
        r = orb.r.to(u.AU).value
        earth = get_body_barycentric_posvel("earth", t)[0].xyz.to(u.AU).value
        itrs = xinglong.get_itrs(t).transform_to(GCRS(obstime=t)).cartesian.xyz.to(u.AU).value
        return r - (earth + itrs)

    p = pos_bary(t_obs + dt_small)
    m = pos_bary(t_obs - dt_small)

    coord_p = SkyCoord(CartesianRepresentation(p * u.AU), frame="icrs")
    coord_m = SkyCoord(CartesianRepresentation(m * u.AU), frame="icrs")

    ra_p, dec_p = coord_p.ra.deg, coord_p.dec.deg
    ra_m, dec_m = coord_m.ra.deg, coord_m.dec.deg

    dra = ra_p - ra_m
    if dra > 180: dra -= 360
    if dra < -180: dra += 360

    ddec = dec_p - dec_m
    dt_day = (2 * dt_small).to(u.day).value

    dra_dt = dra / dt_day
    ddec_dt = ddec / dt_day

    omega_deg_day = np.sqrt((dra_dt*np.cos(np.deg2rad((dec_p+dec_m)/2)))**2 + ddec_dt**2)
    omega_arcsec_hour = omega_deg_day * 3600 / 24

    # ---- append ----
    r_list.append(r_mag.value)
    delta_list.append(delta_mag.value)
    phase_list.append(phase.to(u.deg).value)
    ang_deg_day_list.append(omega_deg_day)
    ang_arcsec_hour_list.append(omega_arcsec_hour)

# ---- 写回 FITS ----
tbl["r_AU"] = r_list
tbl["delta_AU"] = delta_list
tbl["phase_deg"] = phase_list
tbl["ang_rate_deg_day"] = ang_deg_day_list
tbl["ang_rate_arcsec_hour"] = ang_arcsec_hour_list

tbl.write(OUTPUT_FITS, overwrite=True)
print("Wrote:", OUTPUT_FITS)


/opt/anaconda3/envs/jupyter/lib/python3.9/site-packages/astropy/units/decorators.py:313: UserWarning: Wrapping true anomaly to -π <= nu < π
  return_ = wrapped_function(*func_args, **func_kwargs)


Wrote: asteroids_with_params.fits
